# Laboratório do zero — você escreve, o computador confere

**Pedro Henrique G. Modesto** · Mestrado IFSC/USP · orientação: Lucas Madeira

Este notebook junta num arquivo só o conteúdo dos sete notebooks anteriores. A
diferença é que **nenhuma linha de código está escrita**. Cada célula de código tem
só comentários dizendo o que fazer. Você escreve, roda, e confere com o número-alvo.

---

### Como funciona

- Toda célula de código começa com `# ---- ESCREVA AQUI ----` e uma lista numerada.
- Escreva **abaixo** dos comentários. Apague-os depois se quiser.
- Todo exercício tem um **alvo**: um número que tem que aparecer. Se aparecer, está certo.
- Não importe nada de `src/`. Aqui é do zero, com numpy puro.

### A convenção (a mesma do repositório)

$\hbar = m_r = 1$, comprimentos em fm. A equação que resolvemos o tempo todo é

$$u''(r) = 2\,V(r)\,u(r)$$

e a resposta que queremos é o número $a$, o comprimento de espalhamento.

---
# Parte 0 — as ferramentas

Só duas bibliotecas o tempo inteiro.

## Ex 1 · Ligar as ferramentas

**Alvo:** imprimir `pronto`.

In [5]:
# ---- ESCREVA AQUI ----
# 1. importe numpy com o apelido np
# 2. importe matplotlib.pyplot com o apelido plt
# 3. imprima a palavra pronto

import numpy as np
import matplotlib.pyplot as plt

print('pronto')


pronto


## Ex 2 · A grade

Tudo aqui acontece sobre uma lista de distâncias `r`. Duas formas de criar:

- `np.linspace(inicio, fim, n)` — n pontos igualmente espaçados
- `np.arange(n) * dr` — n pontos de passo `dr`

A segunda é a que vamos usar no integrador, porque o passo aparece explícito.

**Alvo:** `r` com 801 pontos, indo de 0 a 8.0, e `r[1] - r[0]` valendo `0.01`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. crie dr = 0.01
# 2. crie n = 801
# 3. crie r = np.arange(n) * dr
# 4. imprima r[0], r[-1] e r[1]-r[0]



dr = 0.01
n = 900

r = np.arange(n) * dr



---
# Parte 1 — o potencial

Antes de resolver qualquer coisa, é preciso dizer ao computador qual é a "cola" entre
os dois átomos. São quatro potenciais no artigo, e todos têm dois parâmetros:
`v` (profundidade) e `mu` (inverso do alcance).

## Ex 3 · O poço esférico

$$V(r) = -v\,\mu^2 \quad (r < R), \qquad V(r) = 0 \quad (r \ge R), \qquad R = 1/\mu$$

O jeito numpy de escrever um "se" que funciona num array inteiro é
`np.where(condicao, valor_se_verdade, valor_se_falso)`.

**Alvo:** `V_poco(np.array([0.5, 1.5]), v=2.0, mu=1.0)` deve dar `[-2., 0.]`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina a função V_poco(r, v, mu)
# 2. dentro dela, calcule R = 1.0/mu
# 3. devolva np.where(r <= R, -v*mu**2, 0.0)
# 4. teste com V_poco(np.array([0.5, 1.5]), 2.0, 1.0)




## Ex 4 · Os outros três

$$V_{\rm gauss} = -v\mu^{2}e^{-r^{2}\mu^{2}}
\qquad
V_{\rm mPT} = \frac{-v\mu^{2}}{\cosh^{2}(\mu r)}
\qquad
V_{\rm LJ} = \tfrac12\!\left(\frac{C_{12}}{r^{12}} - \frac{C_{6}}{r^{6}}\right)$$

O $\tfrac12$ do Lennard-Jones **não** está no artigo — é a correção de convenção que
você descobriu. Sem ele a Tabela 4 não fecha.

**Alvo:** `V_gauss(0.0, 2.0, 1.0)` deve dar `-2.0`, e `V_mpt(0.0, 2.0, 1.0)` também.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina V_gauss(r, v, mu)   -> use np.exp
# 2. defina V_mpt(r, v, mu)     -> use np.cosh
# 3. defina V_lj(r, C12, C6)    -> não esqueça o fator 1/2
# 4. teste os dois primeiros em r = 0.0 com v=2.0, mu=1.0




## Ex 5 · Ver os quatro juntos

**Alvo:** um gráfico com quatro curvas, todas negativas perto da origem e subindo para
zero. O poço tem um degrau vertical; os outros são lisos. Essa diferença vai voltar a
te morder no Ex 14.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. crie rr = np.linspace(0.01, 5, 500)
# 2. abra uma figura com plt.figure(figsize=(7,4))
# 3. plote V_poco(rr, 1.0, 1.0), V_gauss(rr, 1.0, 1.0) e V_mpt(rr, 1.0, 1.0)
#    cada um com plt.plot(rr, ..., label='nome')
# 4. plt.axhline(0, color='k', lw=0.5) para marcar o zero
# 5. plt.xlabel, plt.ylabel, plt.legend, plt.grid(alpha=0.3), plt.show()




---
# Parte 2 — a onda, construída na mão

Agora o coração. A equação $u'' = 2Vu$ vira uma receita de dominó: sabendo dois
pontos, você calcula o próximo.

Da diferença central,
$$u''(r_i) \simeq \frac{u_{i+1} - 2u_i + u_{i-1}}{(\Delta r)^2} = 2V_i u_i$$

isolando:
$$\boxed{u_{i+1} = 2u_i - u_{i-1} + 2(\Delta r)^2 V_i u_i}$$

E o dominó começa com duas peças: `u[0] = 0` (regularidade na origem) e `u[1] = 1`
(a normalização é livre — a equação é linear).

## Ex 6 · O dominó, oito passos, na mão

Antes de automatizar, faça girar devagar e **olhe os números**.

**Alvo:** uma tabela de 10 linhas. Os valores devem crescer, e no fim crescer de forma
quase constante de linha para linha — sinal de que virou reta.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. dr = 0.5
# 2. u_ant = 0.0  e  u_atual = 1.0
# 3. imprima as duas primeiras linhas
# 4. faça um laço  for i in range(2, 10):
#      a) r = i * dr
#      b) V = -2.0 * np.exp(-r**2)          (um gaussiano com v=2, mu=1)
#      c) u_prox = 2*u_atual - u_ant + 2*dr**2 * V * u_atual
#      d) imprima r e u_prox
#      e) u_ant, u_atual = u_atual, u_prox




## Ex 7 · O integrador de verdade

Mesma receita, passo pequeno, guardando tudo num array.

**Alvo:** `u` com 801 valores, `u[0] == 0`, `u[1] == 1`, e um gráfico que sobe
encurvado perto da origem e vira **reta** depois de `r ≈ 3`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina a função integrar(V, dr)
#      V = array com o potencial em cada ponto da grade
#      a) crie u = np.zeros(len(V))
#      b) u[1] = 1.0
#      c) laço  for i in range(1, len(V)-1):
#             u[i+1] = 2*u[i] - u[i-1] + 2*dr**2 * V[i] * u[i]
#      d) devolva u
# 2. crie dr = 0.01 e r = np.arange(801)*dr
# 3. V = V_gauss(r, 1.0, 1.0)
# 4. u = integrar(V, dr)
# 5. imprima u[0] e u[1]
# 6. plote r contra u




---
# Parte 3 — o número $a$

Fora do alcance do potencial $V=0$, então $u''=0$: a solução **é uma reta**. O
comprimento de espalhamento é onde essa reta cruza o zero, extrapolada para dentro.

$$u(r) \propto r - a \qquad (\text{fora do alcance})$$

## Ex 8 · O jeito ingênuo — esticar a régua

Pegue dois pontos bem no fim da onda, ache a reta que passa por eles, e veja onde ela
cruza o zero.

**Alvo:** com o gaussiano `v=1.0, mu=1.0` (o mesmo do Ex 7), deve sair `a ≈ -3.33`.

Negativo: esse poço quase liga o par, mas não liga. O limiar do gaussiano
com `mu=1` fica em `v ≈ 1.342`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. pegue x1, y1 = r[-100], u[-100]   e   x2, y2 = r[-1], u[-1]
# 2. inclinacao = (y2-y1)/(x2-x1)
# 3. a = x1 - y1/inclinacao
# 4. imprima a
# 5. (opcional) plote a onda e a reta esticada, e marque o ponto (a, 0)




## Ex 9 · O jeito certo — derivada logarítmica

O jeito do Ex 8 depende de você escolher "dois pontos bem no fim". Ruim: se o
potencial tiver cauda longa, você escolhe errado.

O jeito certo casa a **derivada logarítmica** de $u$ com a da reta, exatamente na
borda do alcance $R$. Isso não depende da normalização — e é por isso que se usa.

$$\boxed{\;a = R - \frac{2\,\Delta r\; u_N}{u_{N+1} - u_{N-1}}\;}
\qquad\text{(Eq. 110 do artigo)}$$

com $r_N = R$. Para isso a grade precisa ter a borda $R$ **em cima de um ponto** — por
isso o passo é ajustado.

**Alvo:** com o poço `v=0.5, mu=1.0` (logo `R=1`), deve sair `a = -0.557408`, que é
exatamente o valor da fórmula fechada.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina grade(R, dr):
#      a) N = int(np.ceil(R/dr - 1e-12))
#      b) dr_ef = R/N
#      c) r = dr_ef * np.arange(N+2)      <- N+2 pontos: precisa de um ALÉM de R
#      d) devolva r, N, dr_ef
#
# 2. defina extrair_a(r, u, N, dr):
#      devolva r[N] - 2*dr*u[N]/(u[N+1] - u[N-1])
#
# 3. use as duas com o poço v=0.5, mu=1.0:
#      R = 1.0/mu ; r, N, dre = grade(R, 1e-3)
#      u = integrar(V_poco(r, 0.5, 1.0), dre)
#      imprima extrair_a(r, u, N, dre)




## Ex 10 · O teste de fogo — numérico contra a fórmula exata

Para o poço existe fórmula fechada:

$$a = R\left[1 - \frac{\tan x}{x}\right], \qquad x = \sqrt{2v}$$

**Alvo:** as bolinhas do numérico caindo **em cima** da linha do analítico, e a
divergência em $v = \pi^2/8 = 1.2337$, onde nasce o primeiro estado ligado.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina a_poco_exato(v, R): x = np.sqrt(2*v); devolva R*(1 - np.tan(x)/x)
#
# 2. defina a_numerico(v, mu, dr=1e-3):  junte grade + V_poco + integrar + extrair_a
#
# 3. vs = np.linspace(0.1, 3.0, 300)  -> curva analítica
#    vs_pontos = np.linspace(0.1, 3.0, 25)  -> bolinhas numéricas
#
# 4. plote: linha do analítico, bolinhas do numérico (marker='o', mfc='none')
# 5. plt.axvline(np.pi**2/8, ls='--', color='k')
# 6. plt.ylim(-12, 12) para o polo não achatar o gráfico




---
# Parte 4 — o alcance efetivo $r_0$

O segundo número. Ele mede o quanto a onda verdadeira difere da reta extrapolada:

$$r_0 = 2\int_0^R \left[g_0^2(r) - u_0^2(r)\right]dr, \qquad g_0(r) = 1 - \frac{r}{a}$$

Para a conta fazer sentido, $u$ precisa estar **normalizada** de modo que ela vire
exatamente $g_0$ fora do alcance.

## Ex 11 · Normalizar e montar o integrando

A normalização: multiplique $u$ por $C$ tal que $C\,u(R) = g_0(R) = 1 - R/a$.

**Alvo:** um gráfico com $g_0^2$ (tracejado) e $u_0^2$ (cheio), e a área entre as duas
sombreada. Essa área vezes 2 é o $r_0$.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. resolva o gaussiano do dêuteron: v=1.9102, mu=0.6754
#    (para o gaussiano o alcance não é 1/mu; use R = 8.0 e dr = 1e-3 na grade)
# 2. a = extrair_a(...)
# 3. C = (1 - r[N]/a) / u[N]
# 4. un = C*u
# 5. g0 = 1 - r/a
# 6. integrando = g0**2 - un**2
# 7. plote g0**2 e un**2 até r=6, e use plt.fill_between para sombrear entre as duas




## Ex 12 · Trapézio e Simpson, na mão

Duas quadraturas para a mesma integral. Se as duas derem o mesmo número, você
confia; se discordarem, a grade está grossa.

- **Trapézio:** $\Delta r\left[\tfrac12 f_0 + f_1 + \dots + f_{n-2} + \tfrac12 f_{n-1}\right]$
- **Simpson:** $\frac{\Delta r}{3}\left[f_0 + 4f_1 + 2f_2 + 4f_3 + \dots + f_{n-1}\right]$ (n ímpar)

**Alvo:** `a ≈ 5.400` e `r0 ≈ 1.699`. O artigo publica 5.40 e 1.70.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina trapezio(f, dr):
#      devolva dr*(np.sum(f) - 0.5*f[0] - 0.5*f[-1])
#
# 2. defina simpson(f, dr):
#      pesos = np.ones(len(f)); pesos[1:-1:2] = 4; pesos[2:-1:2] = 2
#      devolva dr/3 * np.sum(pesos*f)
#      (se len(f) for par, corte o último ponto antes)
#
# 3. r0_trap = 2*trapezio(integrando[:N+1], dre)
#    r0_simp = 2*simpson(integrando[:N+1], dre)
# 4. imprima a, r0_trap, r0_simp




---
# Parte 5 — o segundo método numérico

O integrador da Parte 2 é de segunda ordem: erro $\mathcal{O}(\Delta r^2)$. Existe um
truque que sobe para quarta ordem quase de graça — o **Numerov**.

Escrevendo $u'' = -\xi(r)u$ com $\xi = -2V$, e chamando $h_2 = (\Delta r)^2/12$:

$$u_{i+1} = \frac{2u_i\left(1 - 5h_2\xi_i\right) - u_{i-1}\left(1 + h_2\xi_{i-1}\right)}
{1 + h_2\,\xi_{i+1}}$$

## Ex 13 · Escrever o Numerov

**Alvo:** com o poço `v=0.5, mu=1.0` e `dr=1e-3`, os dois métodos devem dar
`a = -0.5574` — concordando nas primeiras casas.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina numerov(V, dr):
#      a) xi = -2.0*V
#      b) h2 = dr*dr/12.0
#      c) u = np.zeros(len(V)); u[1] = 1.0
#      d) laço i de 1 até len(V)-2:
#           u[i+1] = (2*u[i]*(1 - 5*h2*xi[i]) - u[i-1]*(1 + h2*xi[i-1])) / (1 + h2*xi[i+1])
#      e) devolva u
#
# 2. rode o poço v=0.5, mu=1.0 com os DOIS métodos e imprima os dois valores de a




## Ex 14 · A surpresa

Agora meça o erro dos dois métodos contra a fórmula exata, para vários `dr`, em
escala log-log. Faça isso para **dois** potenciais: um liso (mPT ou gaussiano) e um
com degrau (o poço).

**Alvo — e é o ponto do exercício:**

| potencial | central | Numerov |
|---|---|---|
| mPT (liso) | ordem 2 · erro `3.6e-07` | **já no piso, `2.8e-08`** |
| poço (degrau) | **ordem 2 · erro `3.5e-07`** | ordem 1 · erro `3.4e-04` |

(erros com `dr = 1e-3`, contra a fórmula exata)

No liso o Numerov ganha fácil — no passo grosso `dr=8e-3` ele é **mil vezes** melhor.
No poço ele perde por **mil vezes**, e a inclinação da reta cai de 2 para 1.

A ordem alta do Numerov **pressupõe** potencial suave. Na borda do poço essa hipótese
morre, e com ela a vantagem. Esse é o Achado nº 2 do seu relatório — e você vai
medi-lo agora.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. drs = np.array([8e-3, 4e-3, 2e-3, 1e-3, 5e-4])
# 2. para cada dr e cada método, calcule o erro relativo |a_num/a_exato - 1|
#    (para o poço use a_poco_exato; para o liso, faça com dr muito pequeno e use como referência)
# 3. plt.loglog(drs, erros, 'o-') para cada método
# 4. dois painéis lado a lado: plt.subplots(1, 2, figsize=(11,4))
# 5. olhe a INCLINAÇÃO das retas: ela é a ordem do método




---
# Parte 6 — universalidade

O fecho. Quatro potenciais com formas completamente diferentes, ajustados para terem
o **mesmo** $a$. Fora do alcance, as quatro soluções têm que virar a **mesma reta**.

Se isso acontecer, você provou com as suas mãos que a baixa energia o formato do
potencial não importa — só dois números importam. É a base da dissertação inteira.

## Ex 15 · As quatro curvas colapsando

Parâmetros do caso nêutron-nêutron (Tabela 3 do artigo), todos com $a \approx -18.5$ fm:

| potencial | v | mu |
|---|---|---|
| mPT | 0.9071 | 0.7991 |
| gaussiano | 1.2121 | 0.5672 |
| poço | 1.1096 | 0.3918 |

**Alvo:** as três curvas separadas perto da origem, e coladas na mesma reta
$1 - r/(-18.5)$ depois de $r \approx 4$ fm.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. para cada um dos três, resolva com R=10, dr=1e-3, e normalize (como no Ex 11)
# 2. plote as três curvas u_normalizada(r) até r=8, com cores diferentes
# 3. plote por cima, tracejada, a reta 1 - r/(-18.5)
# 4. imprima o a de cada um: os três devem bater na primeira casa decimal




## Ex 16 · Juntar tudo numa função

Você tem todas as peças soltas. Agora empacote.

**Alvo:** uma única função que recebe o potencial, o passo e o método, e devolve
`a` e `r0`. Quando ela funcionar, você reescreveu o `src/espalhamento.py` inteiro —
com as suas mãos, sem ter olhado.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina calcular(Vfunc, params, R, dr=1e-3, metodo='numerov')
#      a) r, N, dre = grade(R, dr)
#      b) V = Vfunc(r, *params)
#      c) u = numerov(V, dre) se metodo=='numerov', senão integrar(V, dre)
#      d) a = extrair_a(r, u, N, dre)
#      e) normalize, monte o integrando, e calcule r0 com simpson
#      f) devolva a, r0
#
# 2. teste com o gaussiano do dêuteron: deve dar a=5.400, r0=1.699




---
# Parte 7 — autovalores: quando a energia não é sua

Até aqui a energia era **zero**. Você resolvia e pegava o `a`. Agora vem o outro
lado da mecânica quântica: estados **ligados**, onde a energia é a incógnita.

A equação ganha um termo:

$$u''(x) = 2\left[V(x) - E\right]u(x) \qquad (\hbar = m = 1)$$

E aparece a coisa mais estranha da mecânica quântica: **quase todo `E` é proibido**.
Só um conjunto discreto funciona. Vamos ver por quê — não em palavras, na tela.

## Ex 17 · A armadilha harmônica

$$V(x) = \frac{x^2}{2}$$

É a armadilha dos átomos frios. Literalmente: o potencial óptico que a Patrícia usa é
harmônico perto do fundo.

**Alvo:** a parábola desenhada, e uma grade `x` de −9 a 9.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. x = np.linspace(-9, 9, 4001)   e   dx = x[1]-x[0]
# 2. V_osc = x**2/2
# 3. plote V_osc contra x, com plt.ylim(0, 6)
# 4. desenhe linhas horizontais em E = 0.5, 1.5, 2.5 (plt.axhline)
#    -- são os níveis que você vai ENCONTRAR daqui a dois exercícios




## Ex 18 · Chutar a energia e ver dar errado

Escreva um integrador que aceita `E`. Depois chute um `E` qualquer e olhe o fim da
solução.

**Alvo — e é o ponto:** para `E = 0.3` a solução **explode** para +∞. Para `E = 0.7`
ela explode para −∞. Só entre os dois existe um valor onde ela morre direitinho.

Esse "explodir" é a condição de contorno em $x\to\infty$ te dizendo que o `E` está
errado. É assim que a quantização aparece: não é postulado, é consequência.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina integra_E(V, dx, E):
#      a) u = np.zeros(len(V)) ; u[1] = 1e-10
#      b) laço i de 1 até len(V)-2:
#           u[i+1] = 2*u[i] - u[i-1] + 2*dx**2 * (V[i] - E) * u[i]
#      c) devolva u
#
# 2. para E em [0.3, 0.5, 0.7]:
#      integre e imprima o sinal e a ordem de grandeza de u[-1]
#      (dica: print(E, np.sign(u[-1]), abs(u[-1]))  )
#
# 3. plote as três soluções normalizadas por np.max(np.abs(u)), com plt.ylim(-1.2, 1.2)




## Ex 19 · Caçar o autovalor contando nós

O truque que resolve isso é bonito e é o mesmo do Ex 4 do outro caderno: **não
persiga a explosão, conte os nós.**

Teorema (Sturm): a $n$-ésima autofunção tem exatamente $n$ nós. E o número de nós
**cresce com E**, degrau por degrau. Então: bisseção no número de nós.

**Alvo:** `E_n = n + 0.5`, com erro `< 1e-4`.

| n | E encontrado | exato |
|---|---|---|
| 0 | 0.499999 | 0.5 |
| 1 | 1.499997 | 1.5 |
| 2 | 2.499992 | 2.5 |
| 3 | 3.499984 | 3.5 |

O espaçamento constante de 1 é o $\hbar\omega$. Você acabou de achar na mão o
resultado mais famoso da mecânica quântica.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina conta_nos(u):
#      pegue s = np.sign(u[np.abs(u) > 0])
#      devolva int(np.count_nonzero(s[1:]*s[:-1] < 0))
#
# 2. defina acha_E(V, dx, E_lo, E_hi, n_alvo, iteracoes=60):
#      repita 60 vezes:
#         Em = 0.5*(E_lo + E_hi)
#         se conta_nos(integra_E(V, dx, Em)) > n_alvo:  E_hi = Em
#         senão:                                        E_lo = Em
#      devolva 0.5*(E_lo + E_hi)
#
# 3. para n em range(5): imprima acha_E(V_osc, dx, 0.01, 7.0, n) e compare com n+0.5
#
# CUIDADO: a solução pode estourar o float. Se der overflow, adicione dentro do laço:
#      if abs(u[i+1]) > 1e250: u[:i+2] /= 1e250     (a equação é linear: pode reescalar)




---
# Parte 8 — o átomo de hidrogênio: $n$, $\ell$ e $m$

Agora o sistema que fundou a mecânica quântica. E — não é coincidência — **é a mesma
equação radial do espalhamento**, com dois acréscimos: o potencial de Coulomb e a
barreira centrífuga.

$$u''(r) = 2\left[V_{\rm ef}(r) - E\right]u(r),
\qquad
V_{\rm ef}(r) = -\frac{1}{r} + \frac{\ell(\ell+1)}{2r^{2}}$$

em unidades atômicas ($\hbar = m_e = e = 1$). O termo $\ell(\ell+1)$ é **exatamente** o
que apareceu na Parte I da apostila, e é o mesmo que vira $L^2 - \tfrac14$ na Parte II.

## Ex 20 · O potencial efetivo e a barreira

**Alvo:** três curvas. Para $\ell = 0$ o potencial mergulha para $-\infty$ na origem.
Para $\ell \ge 1$ a barreira centrífuga vence e o potencial sobe para $+\infty$ —
**o elétron não consegue chegar na origem**.

É por isso que só orbitais s têm densidade não nula no núcleo. E é a mesma razão pela
qual só onda-s importa em átomos frios.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina V_hidrogenio(r, l):  devolva -1/r + l*(l+1)/(2*r**2)
# 2. rr = np.linspace(0.05, 25, 2000)
# 3. plote para l = 0, 1, 2
# 4. plt.ylim(-0.6, 0.3) ; plt.axhline(0, color='k', lw=0.5)




## Ex 21 · Os níveis $n$ — a série de Rydberg

Reuse `acha_E` e `conta_nos` do Ex 19, sem mudar nada. Só troque o potencial e a
janela de energia.

**Alvo:** $E_n = -\dfrac{1}{2n^2}$

| n | E encontrado | exato |
|---|---|---|
| 1 | −0.49979 | −0.5 |
| 2 | −0.124974 | −0.125 |
| 3 | −0.055548 | −0.055556 |

Repare que os níveis se **acumulam** em E = 0. Acima de zero começa o contínuo — e o
contínuo é onde vive o espalhamento. O seu laboratório inteiro trabalha ali em cima.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. r = np.linspace(1e-4, 90, 9001)  e  dr = r[1]-r[0]
# 2. V0 = V_hidrogenio(r, 0)
# 3. para k em range(3):  E = acha_E(V0, dr, -0.55, -1e-4, k)
#    (k = número de nós; para l=0, n = k+1)
# 4. imprima E e compare com -0.5/n**2
# 5. desenhe os níveis como linhas horizontais, e marque E=0 como "ionização"




## Ex 22 · O $\ell$ e a degenerescência acidental

Agora com $\ell = 1$. A regra é $n \ge \ell + 1$, então o primeiro nível com $\ell=1$
é o $n=2$.

**Alvo — e é o exercício inteiro:**

$$E(n=2,\ \ell=0) = E(n=2,\ \ell=1) = -0.125$$

Dois estados com formas completamente diferentes, mesma energia. Isso é chamado de
**degenerescência acidental**, e não é acidente nenhum: é uma simetria escondida do
potencial $1/r$ (o vetor de Laplace-Runge-Lenz). Só o Coulomb tem isso — no átomo de
sódio, por exemplo, o 2s e o 2p têm energias diferentes.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. calcule E(n=2, l=0): use V_hidrogenio(r,0) com k=1 nó
# 2. calcule E(n=2, l=1): use V_hidrogenio(r,1) com k=0 nós
# 3. imprima os dois e a diferença
# 4. repita para n=3 com l=0,1,2 -> os TRÊS devem dar -0.0555...




## Ex 23 · E onde está o $m$?

Procure o $m$ na equação que você resolveu. **Ele não está lá.**

Isso não é falha: é o resultado. A separação $\Psi = R(r)\,Y_{\ell m}(\theta,\varphi)$
manda o $m$ inteiro para a parte angular, e a equação radial não o vê. Consequência:

$$E \text{ não depende de } m \quad\Longrightarrow\quad
\text{cada } \ell \text{ tem } 2\ell+1 \text{ estados de mesma energia}$$

$m$ é a projeção do momento angular num eixo. Sem nada quebrando a simetria esférica,
não existe eixo preferido — logo, não pode importar.

**O que quebra:** um campo magnético. Aí entra o Zeeman, $\Delta E \propto m B$, os
$2\ell+1$ níveis se separam, e o $m$ passa a importar.

**E é exatamente por isso que existe ressonância de Feshbach.** Girar `B` move os
níveis com `m` diferente uns em relação aos outros até que um estado molecular cruze
o limiar de colisão — e ali `a` diverge. O botão do experimento da Patrícia é este.

**Alvo:** a contagem de degenerescência batendo com $n^2$.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. para n em range(1, 5):
#      a) some (2*l + 1) para l de 0 até n-1
#      b) imprima n, a soma, e n**2   -> devem ser iguais
#
# 2. (desenho) simule o Zeeman: para l=1, plote E + 0.02*m*B contra B,
#    para m = -1, 0, +1, com B de 0 a 5.
#    Três retas saindo do mesmo ponto: é o m ficando visível.




---
# Parte 9 — números aleatórios e a semente

Muda o assunto, mas não muda o projeto: o método da sua dissertação é **Monte Carlo
quântico**. Antes de VMC e DMC, é preciso ter clareza absoluta sobre uma coisa que
parece burocrática e não é: **de onde vêm os números aleatórios, e como garantir que o
resultado seja reprodutível.**

Um resultado de Monte Carlo que ninguém consegue reproduzir não é resultado.

## Ex 24 · A semente

O computador não tem números aleatórios de verdade. Tem um **gerador
pseudo-aleatório**: um algoritmo determinístico que, a partir de um número inicial (a
**semente**), produz uma sequência que *parece* aleatória.

Mesma semente ⟹ mesma sequência. Sempre. Em qualquer máquina.

**Não use** `np.random.rand()` — ele mexe num estado global escondido, e aí duas
partes do seu código interferem uma na outra sem você ver. **Use** `np.random.default_rng(semente)`,
que te dá um gerador próprio e isolado.

**Alvo:**

```
rng(7): [0.625095 0.897214 0.775686]
rng(7): [0.625095 0.897214 0.775686]   <- idêntico
rng(8): [0.326972 0.987277 0.318711]   <- diferente
```

In [ ]:
# ---- ESCREVA AQUI ----
# 1. rng = np.random.default_rng(7)
# 2. imprima rng.random(3)
# 3. crie OUTRO gerador com a mesma semente 7 e imprima de novo -> idêntico
# 4. crie um com semente 8 -> diferente
#
# 5. REGRA DA CASA, escreva num comentário para não esquecer:
#    toda simulação registra a semente junto com o resultado.
#    Sem a semente, o número não é reproduzível — e não vale como resultado.




## Ex 25 · Integração por Monte Carlo

A ideia mais simples do mundo: para calcular $\int_0^1 f(x)\,dx$, sorteie muitos $x$
uniformes e tire a **média** de $f(x)$.

$$\int_0^1 f(x)\,dx \;\approx\; \frac{1}{N}\sum_{i=1}^{N} f(x_i)$$

Isso é literalmente o esqueleto do VMC: lá você sorteia configurações
$\mathbf{R}$ e tira a média da **energia local** $E_L(\mathbf{R})$.

**Alvo:** $\int_0^1 x^2\,dx = 1/3$. Com `N = 1e6` você deve chegar em ~4 casas.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. rng = np.random.default_rng(42)
# 2. N = 1_000_000
# 3. xs = rng.random(N)          <- sorteia N números uniformes em [0,1)
# 4. estimativa = np.mean(xs**2)
# 5. imprima a estimativa, o valor exato (1/3), e o erro




## Ex 26 · A lei que governa todo Monte Carlo

Agora meça como o erro cai com `N`. Este é **o** número que você precisa ter no
corpo antes de rodar QMC.

**Alvo:** o erro cai como $1/\sqrt{N}$ — a inclinação da reta no log-log é
$-\tfrac12$.

E a consequência prática, que dói: para ganhar **um dígito** de precisão você precisa
de **cem vezes** mais amostras. É por isso que cálculos de QMC rodam em cluster e não
no seu laptop — e por que o Heaviside está no seu projeto.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. Ns = np.array([10**3, 10**4, 10**5, 10**6, 10**7])
# 2. para cada N, repita a estimativa do Ex 25 umas 20 vezes com sementes diferentes
#    e guarde o DESVIO PADRÃO das estimativas (esse é o erro estatístico honesto)
# 3. plt.loglog(Ns, desvios, 'o-')
# 4. plote junto a reta de referência 1/np.sqrt(Ns) (multiplicada por uma constante)
# 5. meça a inclinação: np.polyfit(np.log(Ns), np.log(desvios), 1)[0]  -> deve dar ~ -0.5




---
# Parte 10 — comparando os métodos numéricos

Você já escreveu dois integradores. Agora a pergunta que fecha o caderno: **por que
usar um e não o outro?**

Não existe "o melhor método". Existe o método certo para o problema, e a única forma
honesta de escolher é medindo.

## Ex 27 · O método mais ingênuo que existe: Euler

Antes de comparar os bons, veja o ruim — para ter escala.

Transforme $u'' = 2Vu$ em duas equações de primeira ordem, com $w = u'$:

$$u_{i+1} = u_i + \Delta r\, w_i, \qquad w_{i+1} = w_i + \Delta r\, 2V_i u_i$$

**Alvo:** o Euler é **ordem 1**. Com `dr = 1e-3` ele erra ~1e-3 no `a` do poço,
enquanto a diferença central erra ~3e-7. Mil vezes pior, com o mesmo custo por passo.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina euler(V, dr):
#      u = np.zeros(len(V)); w = np.zeros(len(V))
#      u[0] = 0.0 ; w[0] = 1.0            <- w = derivada; a normalização é livre
#      laço i de 0 até len(V)-2:
#          u[i+1] = u[i] + dr*w[i]
#          w[i+1] = w[i] + dr*2*V[i]*u[i]
#      devolva u
#
# 2. rode o poço v=0.5, mu=1.0 com dr=1e-3 e compare com a_poco_exato(0.5, 1.0)




## Ex 28 · Os três, lado a lado

Agora a medida completa: Euler, diferença central e Numerov, no mesmo problema,
variando `dr`.

**Alvo:** três retas em log-log com inclinações **1, 2 e 4**. A inclinação *é* a ordem
do método — você não vai ler isso num livro, vai medir.

(No poço o Numerov não vai dar 4. Isso não é bug: é o Achado nº 2 aparecendo de novo.
Para ver o 4 de verdade, use o mPT, que é liso.)

In [ ]:
# ---- ESCREVA AQUI ----
# 1. drs = np.array([8e-3, 4e-3, 2e-3, 1e-3, 5e-4])
# 2. para cada método em ('euler', 'central', 'numerov'):
#      calcule o erro relativo do a, para cada dr
# 3. plt.loglog e legenda
# 4. imprima a inclinação de cada um com np.polyfit(np.log(drs), np.log(erros), 1)[0]
# 5. faça DUAS vezes: uma com o poço (degrau) e outra com o mPT (liso)




## Ex 29 · A tabela de decisão

Escreva você a conclusão. Preencha a tabela abaixo com o que **você mediu**, não com
o que eu disse.

| método | ordem medida (liso) | ordem medida (degrau) | custo por passo | quando usar |
|---|---|---|---|---|
| Euler | | | 2 operações | |
| diferença central | | | 3 operações | |
| Numerov | | | ~8 operações | |

**As perguntas que a tabela responde:**

1. Se o potencial é liso e você quer precisão, quem ganha — e por quanto?
2. Se o potencial tem degrau, o método mais caro ainda compensa?
3. O Euler tem alguma situação em que vale a pena? (Pense em quando você precisa da
   *derivada* junto, não só de `u`.)
4. Numerov custa ~3× mais por passo que a diferença central. Para que ordem de
   precisão essa troca compensa?

**A regra geral que sai daqui:** ordem alta pressupõe suavidade. Método caro em
função feia é dinheiro jogado fora. Antes de escolher o método, olhe o potencial.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. junte os números que você mediu no Ex 28 num print organizado
# 2. escreva EM COMENTÁRIO a sua resposta para as 4 perguntas acima
#    (é a parte mais importante do caderno; escreva com as suas palavras)




---

### O que você escreveu

| você escreveu | onde isso vive |
|---|---|
| `V_poco`, `V_gauss`, `V_mpt`, `V_lj` | `src/potenciais.py` |
| `grade`, `integrar`, `numerov`, `euler` | `src/solvers.py` |
| `extrair_a`, `trapezio`, `simpson`, `calcular` | `src/espalhamento.py` |
| `a_poco_exato` | `src/analitico.py` |
| `integra_E`, `conta_nos`, `acha_E` | `src/schrodinger.py` |
| a semente e o erro `1/√N` | a base do VMC/DMC da dissertação |

### Onde procurar os números

Todo valor de literatura que aparece aqui — parâmetros, `a`, `r₀`, de qual artigo veio
e sob quais condições vale — está em **`referencias/literatura.py`**:

```bash
python referencias/literatura.py            # índice
python referencias/literatura.py gauss      # tudo sobre o gaussiano
python referencias/literatura.py deuteron   # tudo sobre o dêuteron
python referencias/literatura.py D1         # a divergência do fator 2
```

### O que vem depois

1. **O arquivo `.py` de verdade** — pegar estas funções, colocar num módulo, e
   escrever os testes que travam cada uma num número analítico.
2. **A segunda série** — Metropolis, energia local, VMC num caso com resposta exata,
   DMC, e o trímero do PRA 2021.

O Ex 26 já plantou a semente disso: você mediu o `1/√N` que governa todo o método da
sua dissertação.